In [1]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from google import genai #type: ignore
from google.genai import types #type: ignore
from google.colab import drive #type: ignore
from dotenv import load_dotenv
import os
import asyncio

drive.mount("/content/drive")
load_dotenv("/content/drive/MyDrive/.env")
api_key = os.getenv("GOOGLE_API_KEY")  

Mounted at /content/drive


In [2]:
client = genai.Client(api_key=api_key)

MODEL_ID = "gemini-2.5-flash"

In [ ]:
from enum import Enum

class IntentEnum(str, Enum):
    """
    Defines the allowed values for the 'intent' field.
    """
    TECHNICAL_SUPPORT = "Technical Support"
    BILLING_INQUIRY = "Billing Inquiry"
    GENERAL_QUESTION = "General Question"

class UserIntent(BaseModel):
    """
    Defines the expected response schema for the intent classification.
    """
    intent: IntentEnum = Field(description="The intent of the user's query")

NameError: name 'Enum' is not defined

In [ ]:
prompt_classification = """
Classify the user's query into one of the following categories.

<categories>
{categories}
</categories>

<user_query>
{user_query}
</user_query>
""".strip()

In [ ]:
def classify_intent(user_query: str) -> IntentEnum:
    """Uses an LLM to classify a user query."""
    prompt = prompt_classification.format(
        user_query=user_query,
        categories=[intent.value for intent in IntentEnum]
    )
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=UserIntent
    )
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=config
    )
    return response.parsed.intent

In [ ]:
prompt_technical_support = """
You are a helpful technical support agent.

Here's the user's query:
<user_query>
{user_query}
</user_query>

Provide a helpful first response, asking for more details like what troubleshooting steps they have already tried.
""".strip()

prompt_billing_inquiry = """
You are a helpful billing support agent.

Here's the user's query:
<user_query>
{user_query}
</user_query>

Acknowledge their concern and inform them that you will need to look up their account, asking for their account number.
""".strip()

prompt_general_question = """
You are a general assistant.

Here's the user's query:
<user_query>
{user_query}
</user_query>

Apologize that you are not sure how to help.
""".strip()

In [ ]:
def handle_query(user_query: str, intent: str) -> str:
    """Routes a query to the correct handler based on its classified intent."""
    if intent == IntentEnum.TECHNICAL_SUPPORT:
        prompt = prompt_technical_support.format(user_query=user_query)
    elif intent == IntentEnum.BILLING_INQUIRY:
        prompt = prompt_billing_inquiry.format(user_query=user_query)
    else:
        prompt = prompt_general_question.format(user_query=user_query)

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )
    return response.text

In [ ]:
query_1 = "My internet connection is not working."

intent_1 = classify_intent(query_1)
response_1 = handle_query(query_1, intent_1)